# 01 - Profiling y dimensiones de calidad

## Objetivo y entorno

Diagnosticar un conjunto de ventas antes de corregir o cargar datos. Todo se ejecuta en **Python local**, con pandas y biblioteca estandar. No se requieren Databricks, Spark, cuentas cloud, SQL Server ni Great Expectations. Instalar las dependencias de `requirements.txt`, seleccionar ese entorno como kernel y ejecutar las celdas de arriba hacia abajo.

Los datos son ficticios: 12 filas, un maestro de 2 productos y una referencia parcial de importes. La fecha de corte es fija para que los resultados no dependan del dia de ejecucion. El modulo `gobierno_demo.py`, en esta misma carpeta, contiene funciones compartidas cuyo codigo debe leerse junto con las celdas. Este notebook no depende de los otros.

## Paso 1 - Observar antes de intervenir

El profiling registra tipos, blancos, nulos y cardinalidad. Una cadena vacia o con espacios puede contar como valor presente para un motor, pero como ausencia para el negocio. El denominador de completitud debe indicar que considera faltante. Contar valores distintos tampoco prueba unicidad de la clave: varias filas pueden repetir el mismo identificador.

Preservaremos `crudos`, incluidos los valores originales, y usaremos `fila_fuente` como identificador tecnico de cada registro para rastrear varios hallazgos sobre la misma fila. En produccion la fuente debe preservarse con controles de acceso y retencion; una copia en memoria no constituye almacenamiento inmutable.

**Preguntas antes de ejecutar:** que columnas parecen claves, que fechas son sospechosas y que moneda contiene espacios? Un formato de email correcto prueba que existe esa casilla? No: solamente permite validar una regla sintactica.

### Organizacion del codigo

La siguiente celda define `perfil` dentro de este notebook porque es una funcion propia de esta demostracion. Se muestran el conteo de blancos, los tipos y el denominador de completitud para poder estudiarlos sin abrir otro archivo.

`gobierno_demo.py` conserva solamente el caso ficticio, la normalizacion y la evaluacion por fila, junto con sus auxiliares y constantes. Ese nucleo es utilizado por los cinco notebooks. Las llamadas a `perfil` que aparecen mas abajo usan la funcion local.

In [1]:
import pandas as pd


def perfil(datos):
    """Describe tipos, faltantes y cardinalidad sin modificar las filas."""
    filas = []
    for columna in datos.columns:
        serie = datos[columna]
        blancos = serie.astype('string').str.strip().eq('').fillna(False)
        faltantes = serie.isna() | blancos
        filas.append({
            'columna': columna,
            'tipo': str(serie.dtype),
            'filas': len(datos),
            'faltantes': int(faltantes.sum()),
            'distintos_no_nulos': int(serie.nunique()),
            'completitud_pct': 100 * (~faltantes).mean() if len(datos) else None,
        })
    return pd.DataFrame(filas)

In [2]:
import sys
from pathlib import Path

candidatos = [Path.cwd(), Path.cwd() / 'notebook' / 'unidad_III']
carpeta = next((ruta for ruta in candidatos if (ruta / 'gobierno_demo.py').exists()), None)
if carpeta is None:
    raise RuntimeError('Ejecutar desde la raiz del repositorio o notebook/unidad_III')
sys.path.insert(0, str(carpeta.resolve()))
import gobierno_demo as demo

crudos, productos, referencia = demo.crear_datos()
display(crudos)
display(perfil(crudos))
assert len(crudos) == 12
assert crudos.loc[0, 'moneda'] == ' ars '

,fila_fuente,id_venta,fecha_venta,id_producto,monto,moneda,email,actualizado_en
0,v01,1,2026-10-05,101,1500.00,ars,uno@demo.invalid,2026-10-06T10:00:00Z
1,v02,2,2026-10-05,102,200.00,ARS,dos@demo.invalid,2026-10-06T10:00:00Z
2,v03,2,2026-10-05,102,250.00,ARS,dos@demo.invalid,2026-10-06T10:00:00Z
3,v04,3,2026-02-29,101,300.00,ARS,tres@demo.invalid,2026-10-06T10:00:00Z
4,v05,4,2026-10-05,101,-10.00,ARS,cuatro@demo.invalid,2026-10-06T10:00:00Z
5,v06,5,2026-10-05,101,500.00,BTC,cinco@demo.invalid,2026-10-06T10:00:00Z
6,v07,6,2026-10-05,999,600.00,ARS,seis@demo.invalid,2026-10-06T10:00:00Z
7,v08,7,2026-10-05,102,700.00,USD,email-invalido,2026-10-06T10:00:00Z
8,v09,8,2026-09-30,102,800.00,ARS,,2026-10-01T10:00:00Z
9,v10,9,2026-10-07,101,900.00,ARS,NaN,2026-10-06T10:00:00Z


,columna,tipo,filas,faltantes,distintos_no_nulos,completitud_pct
0,fila_fuente,str,12,0,12,100.000000
1,id_venta,str,12,1,11,91.666667
2,fecha_venta,str,12,0,4,100.000000
3,id_producto,str,12,0,3,100.000000
4,monto,str,12,0,12,100.000000
5,moneda,str,12,0,4,100.000000
6,email,str,12,3,9,75.000000
7,actualizado_en,str,12,0,2,100.000000


## Paso 2 - Normalizar sin inventar valores

`normalizar` trabaja sobre una copia. Los identificadores se convierten a enteros positivos anulables; un identificador ausente sigue siendo ausente. Los importes se convierten con `Decimal` a centavos, evitando usar un flotante binario para representar dinero. La moneda se recorta y pasa a mayusculas. Se interpretan fechas con formato explicito y zona UTC.

`errors='coerce'` convierte un valor imposible en `NaT`: eso es una senal que una regla debe detectar, no una correccion autorizada. La fecha 2026-02-29 es imposible; no la cambiaremos silenciosamente al 28 de febrero. Tampoco rellenaremos un importe negativo con cero. Debe intervenir quien pueda verificar el valor real.

La salida permite revisar los tipos luego del parseo y localizar las fechas no interpretables. La asercion confirma que el origen conserva su moneda original.

In [3]:
datos = demo.normalizar(crudos)
display(datos[['fila_fuente', 'id_venta', 'fecha_venta', 'id_producto', 'monto_centavos', 'moneda']])
display(perfil(datos))

print("Fechas faltantes o inválidas:", datos['fecha_venta'].isna().sum())
assert datos['fecha_venta'].isna().sum() == 1

print("Moneda original:", repr(crudos.loc[0, 'moneda']))
assert crudos.loc[0, 'moneda'] == ' ars '

print("Moneda normalizada:", datos.loc[0, 'moneda'])
assert datos.loc[0, 'moneda'] == 'ARS'

print("Importe en centavos:", datos.loc[0, 'monto_centavos'])
assert datos.loc[0, 'monto_centavos'] == 150000

,fila_fuente,id_venta,fecha_venta,id_producto,monto_centavos,moneda
0,v01,1,2026-10-05 00:00:00+00:00,101,150000,ARS
1,v02,2,2026-10-05 00:00:00+00:00,102,20000,ARS
2,v03,2,2026-10-05 00:00:00+00:00,102,25000,ARS
3,v04,3,NaT,101,30000,ARS
4,v05,4,2026-10-05 00:00:00+00:00,101,-1000,ARS
5,v06,5,2026-10-05 00:00:00+00:00,101,50000,BTC
6,v07,6,2026-10-05 00:00:00+00:00,999,60000,ARS
7,v08,7,2026-10-05 00:00:00+00:00,102,70000,USD
8,v09,8,2026-09-30 00:00:00+00:00,102,80000,ARS
9,v10,9,2026-10-07 00:00:00+00:00,101,90000,ARS


,columna,tipo,filas,faltantes,distintos_no_nulos,completitud_pct
0,fila_fuente,str,12,0,12,100.000000
1,id_venta,Int64,12,1,10,91.666667
2,fecha_venta,"datetime64[us, UTC]",12,1,3,91.666667
3,id_producto,Int64,12,0,3,100.000000
4,monto,str,12,0,12,100.000000
5,moneda,string,12,0,3,100.000000
6,email,string,12,3,8,75.000000
7,actualizado_en,"datetime64[us, UTC]",12,0,2,100.000000
8,monto_centavos,Int64,12,0,12,100.000000


Fechas faltantes o inválidas: 1
Moneda original: ' ars '
Moneda normalizada: ARS
Importe en centavos: 150000


## Paso 3 - Medir reglas y separar dimensiones

El laboratorio mide completitud, unicidad, validez, consistencia, integridad, vigencia y exactitud. Esta agrupacion didactica conecta con la Unidad III; no implica que exista una lista universal de exactamente seis dimensiones ni una equivalencia normativa uno a uno con ISO/IEC 25012.

- **Completitud:** existe el identificador obligatorio?
- **Unicidad:** hay otra fila con la misma clave? Ambos registros en conflicto se apartan; no se elige uno por su posicion.
- **Validez:** son interpretables la fecha y el importe, y cumplen sus dominios?
- **Consistencia:** la actualizacion es posterior o igual al evento y no ocurre en el futuro?
- **Integridad:** existe el producto y la moneda esta en el dominio autorizado?
- **Vigencia:** la actualizacion tiene como maximo 48 horas respecto de la fecha de corte?
- **Exactitud:** el importe coincide con una referencia independiente disponible?

La exactitud solo puede evaluarse en 2 filas: las otras 10 no tienen evidencia. Eso no las vuelve correctas ni incorrectas. La referencia aqui tambien es ficticia: demuestra el mecanismo, no una certificacion de exactitud real.

La accion es distinta de la dimension: un fallo puede enviar a cuarentena o emitir una advertencia segun el uso aprobado. Los datos de contacto son opcionales para analitica; un email invalido no bloquea necesariamente una venta. Una fila puede tener varios hallazgos. Por eso no se suman los fallos por regla para calcular el numero de filas rechazadas.

In [4]:
aceptadas, rechazadas, metricas, detalle = demo.evaluar(datos, productos, referencia)
display(metricas)
display(detalle)
display(aceptadas[['fila_fuente', 'id_venta']])
assert len(aceptadas) == 4
assert len(rechazadas) == 8
assert len(crudos) == len(aceptadas) + len(rechazadas)
exactitud = metricas.loc[metricas['regla'].eq('DQ_EXACTITUD')].iloc[0]
assert exactitud['evaluados'] == 2
assert exactitud['fallidos'] == 1
assert exactitud['no_evaluables'] == 10
print('Exactitud: 50% en la referencia disponible; cobertura:', round(100 * 2 / len(datos), 2), '%')

,regla,dimension,accion,evaluados,fallidos,no_evaluables,cumplimiento_pct,owner,contrato
0,DQ_ID,completitud,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
1,DQ_UNICIDAD,unicidad,cuarentena,12,3,0,75.000000,Gerencia Comercial,1.0.0
2,DQ_FECHA,validez,cuarentena,12,2,0,83.333333,Gerencia Comercial,1.0.0
3,DQ_MONTO,validez,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
4,DQ_MONEDA,integridad,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
5,DQ_FK_PRODUCTO,integridad,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
6,DQ_ACTUALIZACION,consistencia,cuarentena,12,2,0,83.333333,Gerencia Comercial,1.0.0
7,DQ_EMAIL,validez,advertencia,12,4,0,66.666667,Gerencia Comercial,1.0.0
8,DQ_VIGENCIA,vigencia,advertencia,12,1,0,91.666667,Gerencia Comercial,1.0.0
9,DQ_EXACTITUD,exactitud,advertencia,2,1,10,50.000000,Gerencia Comercial,1.0.0


,fila_fuente,regla,accion
0,v12,DQ_ID,cuarentena
1,v02,DQ_UNICIDAD,cuarentena
2,v03,DQ_UNICIDAD,cuarentena
3,v12,DQ_UNICIDAD,cuarentena
4,v04,DQ_FECHA,cuarentena
5,v10,DQ_FECHA,cuarentena
6,v05,DQ_MONTO,cuarentena
7,v06,DQ_MONEDA,cuarentena
8,v07,DQ_FK_PRODUCTO,cuarentena
9,v04,DQ_ACTUALIZACION,cuarentena


,fila_fuente,id_venta
0,v01,1
7,v08,7
8,v09,8
10,v11,10


Exactitud: 50% en la referencia disponible; cobertura: 16.67 %


## Interpretacion, gobierno y ejercicios

El resultado esperado es **4 filas aptas para continuar y 8 en cuarentena**. Apta no significa perfecta: puede conservar advertencias de email, vigencia o discrepancia con referencia. Tampoco significa que el lote completo este autorizado a publicarse; la puerta de publicacion se estudia en los siguientes notebooks.

El Owner decide aptitud para un uso; el Steward define reglas y revisa casos; el custodio implementa y conserva evidencia. El programador no puede declarar un dato exacto solo porque paso un regex. Los umbrales de 48 horas y el tratamiento del email son decisiones del caso, no exigencias universales de DAMA o ISO.

### Ejercicios

1. Contar faltantes usando solamente `isna()` y comparar con el perfil que incluye blancos. Explicar la diferencia.
2. Inspeccionar las dos filas con `id_venta=2`. Justificar por que no se conserva simplemente la primera.
3. Ampliar la referencia de exactitud con un registro verificado ficticio y comparar cumplimiento y cobertura por separado.
4. Proponer que cambiaria si la finalidad fuese marketing y el email fuera obligatorio. Registrar la propuesta de cambio del contrato sin modificar silenciosamente las reglas originales.

### Limites

La representatividad, la sesgo de muestreo y la verificacion humana no se resuelven con estos indicadores. Antes de automatizar una correccion se necesita una politica, evidencia de negocio y trazabilidad de antes/despues. Consultar el manual de gobierno para RACI, aprobaciones, incidentes y retencion.